# OpenAlex — Scraping Papers and Benchmarking Domain Classifiers

[OpenAlex](https://openalex.org/) is a free, open catalog of over 200 million scholarly works. This notebook uses the OpenAlex REST API to collect academic papers across many disciplines, then compares **multiple classification approaches** for predicting each paper's **knowledge domain** from its title and abstract alone.

**Workflow**

| Step | Task |
|------|------|
| **Collect** | Query OpenAlex with 63 search terms spanning CS, Physics, Biology, Economics, and more |
| **Parse** | Extract the built-in four-level topic labels: *Domain → Field → Subfield → Topic* |
| **Explore** | Visualize class balance, label distributions, and embedding structure (UMAP) |
| **Supervised** | Compare TF-IDF vs. Sentence Transformer features × three classifiers |
| **Domain Embeddings** | Contrast general vs. domain-specific BERT models as frozen feature extractors |
| **Zero-Shot NLI** | Classify without any labeled data using CrossEncoder entailment scoring |
| **LLM (Ollama)** | Prompt a local LLM with a JSON Schema to produce structured domain predictions |
| **Compare** | Leaderboard + pairwise agreement analysis across all methods |
| **BERTopic** | Unsupervised topic discovery — do topics recover supervised labels? |
| **Distill** | Use LLM predictions as pseudo-labels to train a lightweight classical model |

**Prediction target — four OpenAlex knowledge domains:**

| Domain | Example fields |
|--------|----------------|
| Physical Sciences | Computer Science, Physics, Mathematics, Engineering |
| Social Sciences | Sociology, Economics, Political Science, Psychology |
| Life Sciences | Biology, Ecology, Environmental Science, Genetics |
| Health Sciences | Medicine, Nursing, Pharmacology, Public Health |

---

## 1. Imports

Standard libraries plus:
- **scikit-learn** — TF-IDF, three classifiers, train/test split, evaluation metrics
- **imbalanced-learn** — `RandomOverSampler` for class balancing
- **sentence-transformers** — `SentenceTransformer` (dense embeddings) and `CrossEncoder` (NLI zero-shot)
- **transformers** — `pipeline("feature-extraction")` for frozen domain BERT embeddings
- **umap-learn** — UMAP dimensionality reduction for embedding visualization (`pip install umap-learn`)
- **ollama** — local LLM inference with JSON schema–constrained output
- **bertopic** — unsupervised topic modeling (`pip install bertopic`)

Requires Ollama running locally with `ollama pull llama3.2:3b-instruct-q5_K_M`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

import time
import re
import json
import requests
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    accuracy_score, f1_score,
)
import matplotlib.pyplot as plt

import umap
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder
import ollama
from bertopic import BERTopic
from imblearn.over_sampling import RandomOverSampler

pd.set_option('display.max_columns', 100)

## 2. Helper Functions

| Function | Purpose |
|----------|---------|
| `clean_join` | Deduplicate a list and return a comma-separated string (or `NaN`) |
| `normalize_doi` | Strip `https://doi.org/` URL prefixes to return a bare DOI string |
| `reconstruct_abstract` | Reassemble readable text from OpenAlex's inverted-index abstract format |
| `extract_labels` | Pull the four-level topic hierarchy (domain → field → subfield → topic) from a work record |
| `search_openalex` | Call the `/works` endpoint for a single search term and return a list of record dicts |
| `run_all_searches` | Iterate over many terms, deduplicate results, filter short abstracts, and return a clean DataFrame |

In [ ]:
def clean_join(items):
    if not items:
        return np.nan
    cleaned = []
    for x in items:
        if x is None:
            continue
        s = str(x).strip()
        if not s or s.lower() in {"nan", "none", "null"}:
            continue
        if s not in cleaned:
            cleaned.append(s)
    return ", ".join(cleaned) if cleaned else np.nan


def normalize_doi(doi_val):
    if not isinstance(doi_val, str):
        return np.nan
    doi_val = doi_val.strip()
    if not doi_val:
        return np.nan
    doi_val = re.sub(r"^https?://(dx\.)?doi\.org/", "", doi_val, flags=re.IGNORECASE)
    return doi_val if doi_val else np.nan


def reconstruct_abstract(inv_idx):
    """OpenAlex stores abstracts as inverted indexes {word: [positions]}. Reconstruct plain text."""
    if not inv_idx:
        return np.nan
    try:
        max_pos = max(max(lst) for lst in inv_idx.values())
    except Exception:
        return np.nan
    arr = [""] * (max_pos + 1)
    for word, positions in inv_idx.items():
        for p in positions:
            if 0 <= p < len(arr):
                arr[p] = word
    text = " ".join(arr)
    return text if text.strip() else np.nan


def extract_topic_parts(topic):
    if not isinstance(topic, dict):
        return None, None, None, None
    return (
        topic.get("display_name"),
        topic.get("subfield", {}).get("display_name"),
        topic.get("field",    {}).get("display_name"),
        topic.get("domain",   {}).get("display_name"),
    )


def extract_labels(work):
    primary_topic = work.get("primary_topic")
    topics = work.get("topics", []) or []

    topic_names, subfields, fields, domains = [], [], [], []
    all_topics = ([primary_topic] if isinstance(primary_topic, dict) else []) + \
                 [t for t in topics if isinstance(t, dict)]

    for t in all_topics:
        tn, sf, f, d = extract_topic_parts(t)
        if tn: topic_names.append(tn)
        if sf: subfields.append(sf)
        if f:  fields.append(f)
        if d:  domains.append(d)

    primary_name, primary_subfield, primary_field, primary_domain = extract_topic_parts(primary_topic)

    return {
        "domains":          clean_join(domains),
        "fields":           clean_join(fields),
        "subfields":        clean_join(subfields),
        "topics":           clean_join(topic_names),
        "primary_domain":   primary_domain,
        "primary_field":    primary_field,
        "primary_subfield": primary_subfield,
        "primary_topic":    primary_name,
    }


def search_openalex(term, limit, mailto="you@example.com", max_retries=6):
    """
    Query OpenAlex /works for one search term.
    Retries on 429 (rate limit) with exponential backoff: 2, 4, 8, 16, 32, 64 s.
    Set mailto to your real email address to join the polite pool (higher limits).
    """
    url    = "https://api.openalex.org/works"
    params = {"search": term, "per-page": limit, "mailto": mailto}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.RequestException as exc:
            wait = 2 ** (attempt + 1)
            print(f"  Request error ({exc}) — retrying in {wait}s")
            time.sleep(wait)
            continue

        if r.status_code == 200:
            break

        if r.status_code == 429:
            wait = 2 ** (attempt + 1)   # 2 s, 4 s, 8 s, 16 s, 32 s, 64 s
            print(f"  429 rate limit — waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
            continue

        print(f"  OpenAlex error {r.status_code}")
        return []
    else:
        print(f"  Giving up after {max_retries} retries: {term}")
        return []

    results = []
    for w in r.json().get("results", []):
        authors  = [a.get("author", {}).get("display_name") for a in w.get("authorships", [])]
        doi      = normalize_doi(w.get("doi"))
        url_out  = (w.get("primary_location") or {}).get("landing_page_url")
        abstract = reconstruct_abstract(w.get("abstract_inverted_index"))
        labels   = extract_labels(w)
        results.append({
            "search_term": term,
            "doi":         doi,
            "url":         url_out,
            "title":       w.get("title"),
            "authors":     clean_join(authors),
            "abstract":    abstract,
            **labels,
        })
    return results


def run_all_searches(search_terms, limit=10, min_words=25, mailto="you@example.com", sleep=1.0):
    """
    Query OpenAlex for each term; return a deduplicated, filtered DataFrame.
    sleep: seconds between requests (default 1.0 — safe for the anonymous pool).
    Increase to 2.0 if you still see 429s, or set mailto to your email for the polite pool.
    """
    all_results = []
    for i, term in enumerate(search_terms):
        print(f"Searching ({i+1}/{len(search_terms)}): {term}")
        res = search_openalex(term, limit, mailto=mailto)
        print(f"  -> {len(res)} results")
        all_results.extend(res)
        time.sleep(sleep)

    df = pd.DataFrame(all_results)
    if df.empty:
        return df

    for col in df.select_dtypes(include="object"):
        df[col] = df[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df.replace(["nan", "None", "null", ""], np.nan)

    df = df.dropna(subset=["title", "abstract"])
    df = df[df["abstract"].str.split().str.len() >= min_words]

    for col in ["doi", "url", "title"]:
        mask = df[col].notna()
        df = pd.concat(
            [df[mask].drop_duplicates(subset=[col], keep="first"), df[~mask]],
            ignore_index=True,
        )

    ordered_cols = [
        "search_term", "doi", "url", "title", "authors", "abstract",
        "primary_domain", "primary_field", "primary_subfield", "primary_topic",
        "domains", "fields", "subfields", "topics",
    ]
    remaining = [c for c in df.columns if c not in ordered_cols]
    return df[ordered_cols + remaining].reset_index(drop=True)

## 3. Search Terms & Data Collection

We query OpenAlex across 63 topics to build a diverse, multi-domain corpus. Broad coverage ensures all four knowledge domains are well-represented.

**Rate limiting & the polite pool**

OpenAlex enforces a rate limit on anonymous requests. The functions handle this automatically via **exponential backoff** (retries after 2 s, 4 s, 8 s, …), but you can avoid most 429s entirely by passing your real email address in `mailto` — this puts you in OpenAlex's *polite pool* with a significantly higher allowance:

```python
df = run_all_searches(search_terms, limit=50, mailto="your@email.com")
```

**Other API notes:**
- No API key required — OpenAlex is fully open.
- Abstracts come back as an *inverted index* (`{word: [position, ...]}`); `reconstruct_abstract` reassembles them into plain text.
- The default inter-request sleep is **1.0 s** (up from the original 0.2 s). Raise to `sleep=2.0` if you still see 429s without a mailto.
- After collecting, the pipeline drops records missing title or abstract, removes very short abstracts (< 25 words), and deduplicates by DOI, URL, and title.

In [ ]:
search_terms = [

    # --------------------------------------------------
    # Computer Science / AI / Data Science
    # --------------------------------------------------
    "machine learning",
    "deep learning",
    "natural language processing",
    "computer vision",
    "reinforcement learning",
    "graph neural networks",
    "data mining",
    "artificial intelligence ethics",

    # --------------------------------------------------
    # Mathematics / Statistics
    # --------------------------------------------------
    "probability theory",
    "statistical inference",
    "Bayesian statistics",
    "stochastic processes",
    "numerical methods",
    "optimization algorithms",
    "linear algebra applications",

    # --------------------------------------------------
    # Physics
    # --------------------------------------------------
    "quantum mechanics",
    "general relativity",
    "particle physics",
    "astrophysics",
    "cosmology",
    "condensed matter physics",

    # --------------------------------------------------
    # Biology / Medicine
    # --------------------------------------------------
    "genomics",
    "bioinformatics",
    "epidemiology",
    "neuroscience",
    "cancer research",
    "medical imaging",
    "public health analytics",

    # --------------------------------------------------
    # Engineering
    # --------------------------------------------------
    "robotics",
    "control systems",
    "signal processing",
    "wireless communication",
    "renewable energy systems",
    "autonomous vehicles",

    # --------------------------------------------------
    # Psychology / Cognitive Science
    # --------------------------------------------------
    "cognitive psychology",
    "decision making",
    "memory formation",
    "language acquisition",
    "human perception",

]

df = run_all_searches(
    search_terms=search_terms,
    limit=10
)

## 4. Explore the OpenAlex Ontology

OpenAlex organizes all scholarship into a four-level hierarchy:

    Domain  →  Field  →  Subfield  →  Topic

For example, a paper on large language models might be labeled:

    Physical Sciences  →  Computer Science  →  Artificial Intelligence  →  Large Language Models

Each level is progressively more specific. We will use **domain** (four classes) as the classification target — it is the coarsest, most balanced, and most predictable from title and abstract text alone.

The columns in our DataFrame follow this hierarchy:

| Column | Level | Example |
|--------|-------|---------|
| `primary_domain` | Top-level (4 values) | Physical Sciences |
| `primary_field` | Mid-level (~25 values) | Computer Science |
| `primary_subfield` | Fine-grained (~150 values) | Artificial Intelligence |
| `primary_topic` | Most specific (~600+ values) | Ethics and Social Impacts of AI |

In [ ]:
for col, label in [('primary_domain', 'Domain'), ('primary_field', 'Field')]:
    counts = df[col].value_counts()
    print(f"\n-- {label} ({counts.shape[0]} unique values) --")
    print(counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df['primary_domain'].value_counts().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Papers by Domain')
axes[0].set_xlabel('Count')

df['primary_field'].value_counts().head(15).sort_values().plot(
    kind='barh', ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Top 15 Fields')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 5. Text Classification: Predicting Domain from Title + Abstract

We compare **two feature representations** across **three classifiers** — six combinations total — on a held-out stratified test set.

**Feature sets (×2)**

| Feature set | Description |
|-------------|-------------|
| **TF-IDF** | Sparse bag-of-words; up to 15,000 unigrams + bigrams, sublinear TF scaling |
| **SentenceTransformer** | Dense 384-d semantic embeddings from `all-MiniLM-L6-v2` |

**Classifiers (×3)**

| Classifier | Notes |
|------------|-------|
| Logistic Regression | Fast, interpretable linear baseline |
| Linear SVC | Usually the strongest linear text classifier |
| Random Forest | Non-linear ensemble; provides feature importance scores |

**Class balancing** — `RandomOverSampler` duplicates minority-class rows in the *training split only*, so all four domains have equal representation during training. The test split is never modified.

After training, we plot confusion matrices for the best model per feature set, inspect the top TF-IDF features by Random Forest importance, and demonstrate classifying arbitrary text.

In [ ]:
# ── Prepare labeled dataset ────────────────────────────────────────────────
df_cls = df.dropna(subset=['primary_domain', 'title', 'abstract']).copy()
df_cls['text'] = (df_cls['title'].str.strip() + ' ' + df_cls['abstract'].str.strip()).str.strip()

print("Class distribution (raw corpus):")
print(df_cls['primary_domain'].value_counts().to_string())
print(f"\nTotal usable samples: {len(df_cls):,}")

# ── Stratified train / test split (split BEFORE oversampling) ─────────────
X = df_cls['text']
y = df_cls['primary_domain']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Random oversampling — training split only ──────────────────────────────
# RandomOverSampler duplicates minority-class rows at random until all classes
# are equally sized.  It needs a 2-D input, so we reshape the text series.
ros = RandomOverSampler(random_state=42)
X_train_rs, y_train_rs = ros.fit_resample(X_train.values.reshape(-1, 1), y_train)
X_train_rs = pd.Series(X_train_rs.ravel(), name='text')

print("\nAfter oversampling (training set):")
print(pd.Series(y_train_rs).value_counts().to_string())
print(f"\nTrain: {len(X_train_rs):,}  |  Test (unchanged): {len(X_test):,}")

# ── TF-IDF vectorization ───────────────────────────────────────────────────
vectorizer = TfidfVectorizer(max_features=15_000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train_rs)
X_test_tfidf  = vectorizer.transform(X_test)
print(f"\nTF-IDF  : {X_train_tfidf.shape[1]:,} features  |  train matrix {X_train_tfidf.shape}")

# ── Sentence Transformer embeddings ───────────────────────────────────────
# all-MiniLM-L6-v2 produces 384-d dense vectors; ~30 s on CPU for ~2 k docs
st_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nEncoding training texts...")
X_train_st = st_model.encode(X_train_rs.tolist(), batch_size=64, show_progress_bar=True)

print("Encoding test texts...")
X_test_st  = st_model.encode(X_test.tolist(),  batch_size=64, show_progress_bar=True)
print(f"ST embeds: {X_train_st.shape[1]}-d  |  train matrix {X_train_st.shape}")

### 5.1. Embedding Structure (UMAP)

Before training any classifier, project the test-set embeddings into 2-D with **UMAP** to see how well `all-MiniLM-L6-v2` separates the four domains visually. Visible cluster overlap predicts where classifiers will struggle; clean separation means even simple classifiers should do well.

This uses `X_test_st` that was just computed — no extra encoding needed.

In [ ]:
# Project the test-set embeddings to 2-D with UMAP.
# X_test_st is already encoded (no extra compute needed).
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42, verbose=False)
coords  = reducer.fit_transform(X_test_st)

palette = {
    "Physical Sciences": "#4ec9b0",
    "Social Sciences":   "#c586c0",
    "Life Sciences":     "#ce9178",
    "Health Sciences":   "#dcdcaa",
}
fig, ax = plt.subplots(figsize=(9, 7))
for domain, color in palette.items():
    mask = y_test.values == domain
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=color, label=domain, alpha=0.70, s=22, edgecolors="none")
ax.legend(title="Domain", markerscale=2, framealpha=0.3)
ax.set_title("UMAP of all-MiniLM-L6-v2 embeddings (test set, colored by true domain)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

In [ ]:
# ── Define classifiers and feature sets ───────────────────────────────────
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1_000, C=1.0),
    'Linear SVC':          LinearSVC(max_iter=2_000, C=1.0),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
}

feature_sets = {
    'TF-IDF':              (X_train_tfidf, X_test_tfidf),
    'SentenceTransformer': (X_train_st,    X_test_st),
}

# ── Train all 6 combinations ───────────────────────────────────────────────
trained_models = {}
records = []
print("Training (3 models × 2 feature sets = 6 combinations)...\n")

for feat_name, (X_tr, X_te) in feature_sets.items():
    for model_name, clf in classifiers.items():
        clf_fit = clone(clf)
        clf_fit.fit(X_tr, y_train_rs)
        y_pred  = clf_fit.predict(X_te)
        acc     = accuracy_score(y_test, y_pred)
        f1      = f1_score(y_test, y_pred, average='weighted')
        trained_models[(feat_name, model_name)] = clf_fit
        records.append({'Features': feat_name, 'Model': model_name,
                        'Accuracy': round(acc, 4), 'F1 (wtd)': round(f1, 4),
                        'y_pred': y_pred})
        print(f"  {feat_name:<22} | {model_name:<22} | acc={acc:.4f}  f1={f1:.4f}")

# ── Comparison table ───────────────────────────────────────────────────────
summary = (
    pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred'} for r in records])
    .set_index(['Features', 'Model'])
    .sort_values('F1 (wtd)', ascending=False)
)
print("\n── Results ranked by weighted F1 ──")
display(summary)

# ── Confusion matrices: best model per feature set ─────────────────────────
best_per_feat = (
    summary.reset_index()
    .groupby('Features', group_keys=False)
    .apply(lambda g: g.nlargest(1, 'F1 (wtd)'))
    .set_index('Features')['Model']
    .to_dict()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (feat_name, best_model) in zip(axes, best_per_feat.items()):
    rec = next(r for r in records if r['Features'] == feat_name and r['Model'] == best_model)
    ConfusionMatrixDisplay.from_predictions(
        y_test, rec['y_pred'], ax=ax, colorbar=False, xticks_rotation=25
    )
    ax.set_title(f"{feat_name}\n{best_model} (best)")
plt.suptitle('Confusion Matrices — Best Model per Feature Set', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Random Forest feature importance (TF-IDF features) ────────────────────
# Mean-decrease-in-impurity reveals which words / bigrams most strongly
# distinguish the four knowledge domains in the TF-IDF + RF model.
rf_tfidf      = trained_models[('TF-IDF', 'Random Forest')]
feature_names = vectorizer.get_feature_names_out()
importances   = rf_tfidf.feature_importances_

top_n   = 25
top_idx = np.argsort(importances)[-top_n:]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(feature_names[top_idx], importances[top_idx], color='steelblue', edgecolor='white')
ax.set_title(f'Top {top_n} TF-IDF Features — Random Forest Importance')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

# ── Classify new text with the best pipeline ──────────────────────────────
# Default: SentenceTransformer + Logistic Regression.
# Swap to any key in trained_models to try a different combination.
best_clf = trained_models[('SentenceTransformer', 'Logistic Regression')]

def predict_domain(text: str) -> str:
    """Return the predicted OpenAlex knowledge domain for a free-text string."""
    return best_clf.predict(st_model.encode([text]))[0]

examples = [
    ("Physical Sciences",
     "A transformer architecture for graph-structured data achieves SOTA on molecular property prediction."),
    ("Social Sciences",
     "Panel data from 42 countries estimates the effect of carbon taxation on household income inequality."),
    ("Life Sciences",
     "CRISPR-Cas9 knockout of BRCA1 in HEK293T cells significantly reduced homologous recombination efficiency."),
    ("Health Sciences",
     "Randomized controlled trial of cognitive-behavioral therapy versus SSRIs for treatment-resistant depression."),
    ("Physical Sciences",
     "Quantum entanglement in a photonic crystal cavity cooled to millikelvin temperatures via dilution refrigerator."),
]

print(f"{'Predicted':<22}  {'Expected':<22}  Text (truncated)")
print("-" * 100)
for expected, text in examples:
    predicted = predict_domain(text)
    match = "OK" if predicted == expected else "!!"
    print(f"[{match}] {predicted:<22}  {expected:<22}  {text[:62]}...")

## 6. Domain Embeddings: Does the Pretraining Corpus Match?

The existing `all-MiniLM-L6-v2` embeddings are general-purpose. Here we swap in **frozen domain-specific BERT models** and ask: does a pretraining corpus that matches the text (scientific papers) improve classification?

Three models — same frozen feature-extraction pipeline, same LogReg head, same evaluation:

| Model | Pretraining corpus | Expected match to OpenAlex abstracts? |
|---|---|---|
| `bert-base-uncased` | General English | Baseline |
| `allenai/scibert_scivocab_uncased` | ~3M Semantic Scholar papers (science) | ✓ Matched |
| `emilyalsentzer/Bio_ClinicalBERT` | MIMIC-III clinical notes (EHR) | ✗ Mismatched |

**Prediction:** SciBERT gains over BERT-base; Bio_ClinicalBERT loses (clinical vocabulary ≠ academic abstract vocabulary). The gap between them quantifies the value of *domain match*, not just *domain pretraining* in general.

Embeddings use the `transformers` `feature-extraction` pipeline with mean pooling — the same pattern as `U3-1_NLP-8_ClinicalBERT.ipynb`. A capped balanced subset is used so embedding runs on CPU during class.

In [ ]:
GENERAL_MODEL  = "bert-base-uncased"
SCIBERT_MODEL  = "allenai/scibert_scivocab_uncased"
CLINICAL_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
MAX_EMB_LEN    = 128   # truncate for CPU speed

def embed_texts_hf(model_name, texts, max_len=MAX_EMB_LEN):
    """HuggingFace feature-extraction pipeline → mean-pool → (N, hidden_dim) array."""
    pipe = hf_pipeline("feature-extraction", model=model_name)
    vecs = []
    for i, t in enumerate(texts):
        out = np.array(pipe(t, tokenize_kwargs={"truncation": True, "max_length": max_len}))[0]
        vecs.append(out.mean(axis=0))
        if (i + 1) % 50 == 0:
            print(f"  [{model_name.split('/')[-1]}] {i+1}/{len(texts)}")
    return np.vstack(vecs)

# ── Balanced subset — cap per class so CPU embedding is classroom-runnable ──
MAX_PER_CLS = 50   # 50 × 4 = 200 papers; raise if you have a GPU
np.random.seed(42)
sub_idx = (
    df_cls.groupby("primary_domain", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_CLS), random_state=42))
    .index
)
df_sub    = df_cls.loc[sub_idx].reset_index(drop=True)
texts_sub = df_sub["text"].tolist()
y_sub     = df_sub["primary_domain"].to_numpy()
print(f"Domain-embedding subset: {len(df_sub)} papers")
print(pd.Series(y_sub).value_counts().to_string())

idx_sub = np.arange(len(df_sub))
idx_str, idx_ste, y_str, y_ste = train_test_split(
    idx_sub, y_sub, test_size=0.25, stratify=y_sub, random_state=42
)
y_emb_te = y_ste   # stored for leaderboard in §9
print(f"\nTrain={len(idx_str)}  Test={len(idx_ste)}")

# ── Embed with each model and evaluate ────────────────────────────────────
domain_emb_results = []
for model_name in [GENERAL_MODEL, SCIBERT_MODEL, CLINICAL_MODEL]:
    short = model_name.split("/")[-1]
    print(f"\nEmbedding with {short} …")
    E = embed_texts_hf(model_name, texts_sub)

    sc   = StandardScaler()
    E_tr = sc.fit_transform(E[idx_str])
    E_te = sc.transform(E[idx_ste])

    clf  = LogisticRegression(max_iter=3000, class_weight="balanced")
    clf.fit(E_tr, y_str)
    pred = clf.predict(E_te)
    acc  = accuracy_score(y_ste, pred)
    f1   = f1_score(y_ste, pred, average="macro", zero_division=0)
    print(f"  acc={acc:.3f}  macro-F1={f1:.3f}")
    domain_emb_results.append({"name": short, "acc": acc, "f1": f1})

# ── Grouped bar chart ──────────────────────────────────────────────────────
names = [r["name"] for r in domain_emb_results]
accs  = [r["acc"]  for r in domain_emb_results]
f1s   = [r["f1"]   for r in domain_emb_results]

x = np.arange(len(names))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, accs, w, label="accuracy", color="#4ec9b0")
ax.bar(x + w/2, f1s,  w, label="macro-F1", color="#c586c0")
ax.set_xticks(x, names, rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Same classifier, three embedding models\n(frozen feature extractors — no fine-tuning)")
ax.legend()
for xi, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(xi - w/2, a + 0.01, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(xi + w/2, f + 0.01, f"{f:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Zero-Shot Classification via CrossEncoder NLI

**Natural Language Inference (NLI)** frames classification as entailment scoring: for each paper, score how strongly the text *entails* each candidate hypothesis (e.g., `"This paper is about life sciences."`). No training data. No fine-tuning. Just a pre-trained NLI model and plain-English category descriptions.

Model: `cross-encoder/nli-deberta-v3-base`.

**When this shines:** categories are describable in plain English, no labeled data exists yet, or you need a fast baseline before collecting annotations. The four OpenAlex domain names are already descriptive — a good fit.

In [ ]:
# Domain names match the OpenAlex ontology exactly — used in §7, §8, §9
DOMAINS = ["Physical Sciences", "Social Sciences", "Life Sciences", "Health Sciences"]

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
print(f"Loading CrossEncoder: {NLI_MODEL} …")
nli_model = CrossEncoder(NLI_MODEL)

# Hypotheses: one sentence per domain
HYPOTHESES = [f"This paper is about {d.lower()}." for d in DOMAINS]

def classify_nli(text: str) -> str:
    """Score entailment for each domain hypothesis; return the argmax domain."""
    pairs  = [[text[:512], hyp] for hyp in HYPOTHESES]
    logits = nli_model.predict(pairs, apply_softmax=True)
    # deberta-v3 NLI label order: [contradiction, entailment, neutral]
    try:
        id2label = nli_model.config.id2label
        ent_idx  = next(k for k, v in id2label.items() if "entail" in v.lower())
    except Exception:
        ent_idx = 1   # deberta-v3 default
    return DOMAINS[int(np.argmax(logits[:, ent_idx]))]

# ── Run zero-shot NLI on the full test set ────────────────────────────────
print("Running zero-shot NLI classification on test set…")
nli_preds = []
for i, text in enumerate(X_test.tolist()):
    nli_preds.append(classify_nli(text))
    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(X_test)}]")

acc_nli = accuracy_score(y_test, nli_preds)
f1_nli  = f1_score(y_test, nli_preds, average="macro", zero_division=0)
print(f"\nCrossEncoder NLI (zero-shot)  acc={acc_nli:.3f}  macro-F1={f1_nli:.3f}")
print(classification_report(y_test, nli_preds, zero_division=0))

## 8. LLM Classification: Ollama + JSON Schema

Pass each paper's title and abstract to a locally-running LLM and ask it to return a structured prediction. The model is given a **JSON Schema** that constrains its output to exactly one of the four domain strings — no parsing heuristics needed.

This approach requires **no training data** and can handle categories defined on-the-fly in the prompt. The trade-off: it is much slower than a trained classifier and depends on the LLM already knowing what these domains mean.

**Requirement:** `ollama pull llama3.2:3b-instruct-q5_K_M`

A stratified sample of the test set is used (Ollama processes ~1–2 s/paper on CPU; GPU is ~5–10× faster). The sample and predictions are reused by §9 (comparison) and §11 (distillation).

In [ ]:
# DOMAINS defined in §7; reuse here
DOMAIN_SCHEMA = {
    "type": "object",
    "properties": {
        "domain": {
            "type": "string",
            "enum": DOMAINS,
            "description": "The single OpenAlex knowledge domain that best describes this paper",
        }
    },
    "required": ["domain"],
    "additionalProperties": False,
}

SYSTEM_MSG = (
    "You are a domain classifier. Given a paper title and abstract, "
    "return the single OpenAlex knowledge domain that best describes it. "
    f"Choose exactly one of: {', '.join(DOMAINS)}."
)

# ── Stratified sample from X_test (Ollama is slow — ~1–2 s/paper on CPU) ──
OLLAMA_N = 12   # per domain → 48 papers total; raise to 15 for fuller results
np.random.seed(42)
sample_idx = []
for domain in sorted(y_test.unique()):
    pool   = y_test[y_test == domain].index.tolist()
    chosen = list(np.random.choice(pool, size=min(OLLAMA_N, len(pool)), replace=False))
    sample_idx.extend(chosen)

X_sample = X_test.loc[sample_idx]
y_sample  = y_test.loc[sample_idx]
print(f"Ollama sample: {len(X_sample)} papers")
print(pd.Series(y_sample).value_counts().to_string())

# ── Run Ollama with JSON Schema ────────────────────────────────────────────
ollama_preds = []
for i, (_, text) in enumerate(X_sample.items()):
    resp = ollama.chat(
        model="llama3.2:3b-instruct-q5_K_M",
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": f"Title and abstract:\n{text[:1200]}"},
        ],
        format=DOMAIN_SCHEMA,
        options={"temperature": 0},
    )
    try:
        pred = json.loads(resp["message"]["content"])["domain"]
    except Exception:
        pred = "Physical Sciences"   # safe fallback
    ollama_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(X_sample)}] classified")

print(f"\nOllama  acc={accuracy_score(y_sample, ollama_preds):.3f}"
      f"  macro-F1={f1_score(y_sample, ollama_preds, average='macro', zero_division=0):.3f}")

## 9. Cross-Method Comparison

Two views of how the methods stack up:

1. **Leaderboard** — accuracy + macro-F1 for every method.
   *Caveat:* methods were not all evaluated on the same corpus slice (§5/§7 used the full 205-paper test set; §6 used a balanced 50-per-class subset; §8 used a small stratified sample), so cross-section ranking is indicative rather than conclusive.

2. **Agreement matrix** — restricting to the Ollama sample where *all* three methods have predictions, pairwise agreement rates reveal which methods share failure modes and which go their own way. Papers where all three methods disagree are the most pedagogically interesting.

In [ ]:
# ── Leaderboard: all methods on their evaluation sets ─────────────────────
leaderboard_rows = []

# Section 5 — supervised classifiers (full test set)
for rec in records:
    leaderboard_rows.append({
        "Section": "S5",
        "Method":  f"{rec['Features']} + {rec['Model']}",
        "Type":    "Supervised",
        "Eval n":  len(y_test),
        "Accuracy": rec["Accuracy"],
        "Macro-F1": round(f1_score(y_test, rec["y_pred"], average="macro", zero_division=0), 4),
    })

# Section 6 — domain embeddings (balanced subset)
for res in domain_emb_results:
    leaderboard_rows.append({
        "Section": "S6",
        "Method":  f"{res['name']} + LogReg",
        "Type":    "Supervised (frozen)",
        "Eval n":  len(y_emb_te),
        "Accuracy": round(res["acc"], 4),
        "Macro-F1": round(res["f1"], 4),
    })

# Section 7 — NLI zero-shot (full test set)
leaderboard_rows.append({
    "Section": "S7",
    "Method":  "CrossEncoder NLI",
    "Type":    "Zero-shot",
    "Eval n":  len(y_test),
    "Accuracy": round(accuracy_score(y_test, nli_preds), 4),
    "Macro-F1": round(f1_score(y_test, nli_preds, average="macro", zero_division=0), 4),
})

# Section 8 — Ollama (sample)
leaderboard_rows.append({
    "Section": "S8",
    "Method":  "Ollama llama3.2:3b",
    "Type":    "Zero-shot LLM",
    "Eval n":  len(y_sample),
    "Accuracy": round(accuracy_score(y_sample, ollama_preds), 4),
    "Macro-F1": round(f1_score(y_sample, ollama_preds, average="macro", zero_division=0), 4),
})

lb = (pd.DataFrame(leaderboard_rows)
      .sort_values("Macro-F1", ascending=False)
      .reset_index(drop=True))
print("── Method leaderboard (sorted by Macro-F1) ──")
print("  Note: S5/S7 use the full 205-paper test set; S6 uses a 50-per-class balanced")
print("  subset; S8 uses a small stratified sample. Eval sets differ — rank is indicative.\n")
display(lb)

# ── Pairwise agreement on the shared Ollama sample ────────────────────────
sample_positions = X_test.index.get_indexer(X_sample.index)
s5_best = next(r["y_pred"] for r in records
               if r["Features"] == "SentenceTransformer"
               and r["Model"] == "Logistic Regression")

agree_dict = {
    "S5 ST+LR":   np.array(s5_best)[sample_positions],
    "S7 NLI":     np.array(nli_preds)[sample_positions],
    "S8 Ollama":  np.array(ollama_preds),
}
agree_df = pd.DataFrame(agree_dict)
methods  = list(agree_dict.keys())

mat = pd.DataFrame(
    [[(agree_df[m1] == agree_df[m2]).mean() for m2 in methods] for m1 in methods],
    index=methods, columns=methods,
)
print(f"\n── Pairwise agreement on Ollama sample (n={len(agree_df)}) ──")
display(mat.round(3))

# Surface papers where all three methods disagree
disagree_pos = np.where(agree_df.nunique(axis=1) == 3)[0]
print(f"\nPapers where all three methods give different predictions: {len(disagree_pos)}")
if len(disagree_pos) > 0:
    show = pd.DataFrame({
        "true":       y_sample.iloc[disagree_pos].values,
        "S5 ST+LR":  agree_dict["S5 ST+LR"][disagree_pos],
        "S7 NLI":    agree_dict["S7 NLI"][disagree_pos],
        "S8 Ollama": agree_dict["S8 Ollama"][disagree_pos],
        "text":      X_sample.iloc[disagree_pos].str[:90].values + "…",
    }).head(5)
    display(show)

## 10. Unsupervised Contrast: BERTopic

All previous sections used the true `primary_domain` labels. **BERTopic** discovers thematic clusters from the text itself — no labels at all. Comparing its discovered topics to the ground-truth domains answers:

> *Would we have arrived at a similar taxonomy without any human annotation?*

If topics align cleanly with domains, the label taxonomy was (in retrospect) recoverable from text structure alone. If topics cut across domains, it reveals finer-grained scientific themes that the four-label taxonomy hides.

`pip install bertopic`

In [ ]:
# ── BERTopic: unsupervised topic discovery on the full corpus ───────────────
texts_bt  = df_cls["text"].tolist()
y_true_bt = df_cls["primary_domain"].tolist()

vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    min_topic_size=15,
    nr_topics=12,        # merge down to ~12 topics
    verbose=False,
)
topics, _ = topic_model.fit_transform(texts_bt)

n_topics = topic_model.get_topic_info().shape[0] - 1  # exclude outlier -1
print(f"Topics discovered (after merging): {n_topics}")
print(topic_model.get_topic_info().head(8).to_string(index=False))

# ── Cross-tab: topics vs. true domain labels ───────────────────────────────
df_bt = pd.DataFrame({
    "domain": y_true_bt,
    "topic":  [f"T{t:+d}" for t in topics],
})
ct = pd.crosstab(df_bt["topic"], df_bt["domain"], normalize="index").round(2)
ct = ct.sort_values("Physical Sciences", ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(ct) * 0.45)))
ct.plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
ax.set_title("BERTopic topics vs. true domain labels  (row-normalized proportions)")
ax.set_xlabel("Proportion of papers in topic")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

print("\nTopics dominated by a single domain ≈ strong topic↔domain alignment.")
print("Topics mixing multiple domains reveal themes that cut across the four-label taxonomy.")

## 11. Distillation: LLM as Pseudo-Labeler *(optional capstone)*

The Ollama predictions from §8 are **noisy labels** produced without any human annotation. A practical trick: use those pseudo-labels to train a tiny, fast sklearn model on the same sentence embeddings. The trained model then classifies *without* an LLM at inference time — Ollama is only needed once, at labeling time.

This is a lightweight form of **knowledge distillation**: the LLM "teaches" a classical model by generating its own supervision signal.

**Teaching question:** How much performance does the distilled model lose compared to one trained on true labels? The gap quantifies the *cost of skipping annotation*.

In [ ]:
# ── Distillation: train on Ollama pseudo-labels, test on true labels ────────
# The Ollama predictions (no gold labels used) become the training signal for
# a lightweight LogReg on top of the already-computed all-MiniLM embeddings.
# We train on the Ollama sample and evaluate on the REMAINING test papers.

_sample_pos = X_test.index.get_indexer(X_sample.index)
_remain_pos  = np.setdiff1d(np.arange(len(X_test)), _sample_pos)

E_tr_d  = X_test_st[_sample_pos]      # embeddings of Ollama-labeled papers
E_te_d  = X_test_st[_remain_pos]      # embeddings of held-back papers
y_tr_d  = np.array(ollama_preds)      # Ollama predictions as training labels
y_te_d  = y_test.iloc[_remain_pos].to_numpy()   # true labels (evaluation only)

sc_d   = StandardScaler()
clf_d  = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_d.fit(sc_d.fit_transform(E_tr_d), y_tr_d)
pred_d = clf_d.predict(sc_d.transform(E_te_d))

acc_d = accuracy_score(y_te_d, pred_d)
f1_d  = f1_score(y_te_d, pred_d, average="macro", zero_division=0)
print(f"Distilled model")
print(f"  Trained on:  {len(E_tr_d)} papers with Ollama pseudo-labels (no true labels used)")
print(f"  Tested on:   {len(y_te_d)} papers with true labels")
print(f"  acc={acc_d:.3f}  macro-F1={f1_d:.3f}")

# Compare to the gold-supervised ST+LR baseline from Section 5
_gold_pred = next(r["y_pred"] for r in records
                  if r["Features"] == "SentenceTransformer"
                  and r["Model"] == "Logistic Regression")
f1_gold = f1_score(y_test, _gold_pred, average="macro", zero_division=0)
print(f"\nGold ST + LogReg (Section 5, trained on {len(X_train_rs)} true labels)  macro-F1={f1_gold:.3f}")
print(f"Cost of skipping annotation:  Δ macro-F1 = {f1_gold - f1_d:+.3f}")

## 12. When to Use Each Approach

| Approach | Needs labels? | Compute | Domain match needed? | Best when… |
|---|---|---|---|---|
| TF-IDF + classifier | ✓ | Low | No | Labels available; interpretability matters; vocabulary is distinctive |
| General embeddings + classifier | ✓ | Medium | No | Labels available; semantic similarity helps beyond keywords |
| Domain embeddings + classifier | ✓ | Medium | Yes | Labels available + text matches pretraining domain |
| Zero-shot NLI | ✗ | Medium | No | No labels; categories are describable in plain English |
| LLM (Ollama) | ✗ | High | No | No labels; categories are complex or require reasoning |
| BERTopic | ✗ | Medium | No | No labels; goal is to *discover* what the categories are |
| Distillation (LLM → small model) | Pseudo only | Medium | No | No gold labels but need a fast deployed classifier |

**Key takeaway:** The right method depends on what you have (labeled data? GPU?) and what you need (interpretability? coverage of unseen categories?). No method dominates across all settings — understanding the tradeoffs is the goal.

## Review

Seven ways to attach a knowledge domain to a paper, benchmarked on one scraped corpus.

**Takeaways**

- **The supervised baselines are hard to beat in-domain.** TF-IDF or sentence embeddings with a linear head is quick, cheap, and strong whenever you have labels.
- **Domain-matched pretraining helps; domain-mismatched pretraining hurts.** SciBERT gains on academic abstracts and a clinical model loses on them, which is the sharper version of "use a domain model" - it is the *match* that matters, not the specialisation.
- **Zero-shot NLI and LLM prompting need no labels at all**, which makes them the right first move on a brand-new problem and the wrong long-term answer for anything high volume.
- **Distillation is how you keep the zero-shot capability without the zero-shot bill** - let the expensive model label, then train something small on its output.
- **OpenAlex's own topic hierarchy is the label source here**, so the "ground truth" is itself an automated classification. Worth remembering before reading any accuracy number as absolute.